# 🔄 Grundläggande Agentarbetsflöden med GitHub-modeller (Python)

## 📋 Handledning för arbetsflödesorkestrering

Denna anteckningsbok introducerar de kraftfulla **Workflow Builder**-funktionerna i Microsoft Agent Framework. Lär dig hur du skapar sofistikerade, flerstegiga agentarbetsflöden som kan hantera komplexa affärsprocesser och sömlöst koordinera flera AI-operationer.

## 🎯 Lerningsmål

### 🏗️ **Arbetsflödesarkitektur**
- **Workflow Builder**: Designa och orkestrera komplexa flerstegiga processer
- **Händelsestyrd exekvering**: Hantera arbetsflödeshändelser och tillståndsövergångar
- **Visuell arbetsflödesdesign**: Skapa och visualisera arbetsflödesstrukturer
- **GitHub Models-integration**: Utnyttja AI-modeller inom arbetsflödeskontekster

### 🔄 **Processorkestrering**
- **Sekventiella operationer**: Kjedja flera agentuppgifter i logisk ordning
- **Villkorlig logik**: Implementera beslutspunkter och förgrenande arbetsflöden
- **Felhanteing**: Robust felåterställning och arbetsflödesmotståndskraft
- **Tillståndshantering**: Spåra och hantera arbetsflödets exekveringstillstånd

### 📊 **Företagsarbetsflödesmönster**
- **Automatisering av affärsprocesser**: Automatisera komplexa organisatoriska arbetsflöden
- **Multi-agentkoordinering**: Koordinera flera specialiserade agenter
- **Skalbar exekvering**: Designa arbetsflöden för företagsomfattande operationer
- **Övervakning & observabilitet**: Spåra arbetsflödets prestanda och resultat

## ⚙️ Förutsättningar & installation

### 📦 **Nödvändiga beroenden**

Installera Agent Framework med arbetsflödesfunktioner:

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub Models-konfiguration**

**Miljöinställning (.env-fil):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 **Företagsanvändningsfall**

**Exempel på affärsprocesser:**
- **Kundintroduktion**: Flerstegsverifiering och installationsarbetsflöden
- **Innehållspipeline**: Automatiserad innehållsskapande, granskning och publicering
- **Databehandling**: ETL-arbetsflöden med AI-driven transformation
- **Kvalitetssäkring**: Automatiska test- och valideringsprocesser

**Arbetsflödets fördelar:**
- 🎯 **Pålitlighet**: Deterministisk exekvering med felåterställning
- 📈 **Skalbarhet**: Hantera automatisering av processer med hög volym
- 🔍 **Observabilitet**: Kompletta revisionsspår och övervakning
- 🔧 **Underhållbarhet**: Visuell design och modulära komponenter

## 🎨 Arbetsflödesdesignmönster

### Grundstruktur för arbetsflöde
```mermaid
graph TD
    A[Start] --> B[Agent Uppgift 1]
    B --> C{Beslutspunkt}
    C -->|Success| D[Agent Uppgift 2]
    C -->|Failure| E[Felhantskare]
    D --> F[Slut]
    E --> F
```

**Nyckelkomponenter:**
- **WorkflowBuilder**: Huvudorkestreringsmotor
- **WorkflowEvent**: Händelsehantering och kommunikation
- **WorkflowViz**: Visuell arbetsflödesrepresentation och felsökning

Låt oss bygga ditt första intelligenta arbetsflöde! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
